# PDAC vs non-PDAC from urinary biomarkers — leak-free evaluation

**Task.** Binary classification: pancreatic ductal adenocarcinoma (diagnosis = 3) vs everyone else (healthy controls and benign hepatobiliary conditions), using the public data of Debernardi et al., *PLOS Medicine* 2020.

**What changed (Oct 2026).** An earlier version of this notebook duplicated the PDAC rows *before* the train/test split and fitted the imputer on the full dataset, so identical rows appeared in both train and test and the reported recall (0.97) was inflated. This notebook:

1. never duplicates rows; class imbalance is handled with `class_weight`,
2. fits imputation and scaling inside a `Pipeline`, so they only ever see training folds,
3. reports repeated stratified 5-fold cross-validation (5 folds × 3 repeats) with mean ± std,
4. compares two feature sets: the urine panel alone vs urine panel + plasma CA19-9,
5. reproduces the old protocol at the end, to show the size of the leak.

This is a learning exercise on a small public dataset (590 patients), not a diagnostic tool.

In [1]:
import numpy as np, pandas as pd, warnings
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import make_scorer, recall_score, accuracy_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
SEED = 42

In [2]:
df = pd.read_csv('Debernardi et al 2020 data.csv')
df['y'] = (df['diagnosis'] == 3).astype(int)
df['sex'] = df['sex'].map({'M': 1, 'F': 0})
print(df.shape)
print(df['y'].value_counts().rename({1: 'PDAC', 0: 'not PDAC'}))
print('Missing plasma_CA19_9:', df['plasma_CA19_9'].isna().sum())

(590, 15)
y
not PDAC    391
PDAC        199
Name: count, dtype: int64
Missing plasma_CA19_9: 240


## Feature sets

- **Urine panel**: age, sex, creatinine, LYVE1, REG1B, TFF1 (complete for every patient).
- **Urine + CA19-9**: adds plasma CA19-9, a blood marker that is missing for 240 patients and is imputed inside each training fold.

`stage` and `benign_sample_diagnosis` are excluded because they encode the label.

In [3]:
URINE = ['age', 'sex', 'creatinine', 'LYVE1', 'REG1B', 'TFF1']
WITH_CA199 = URINE + ['plasma_CA19_9']
y = df['y']

def models():
    return {
        'Logistic regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                             LogisticRegression(class_weight='balanced', max_iter=1000)),
        'Random forest': make_pipeline(SimpleImputer(strategy='median'),
                                       RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED)),
        'LightGBM': make_pipeline(SimpleImputer(strategy='median'),
                                  LGBMClassifier(class_weight='balanced', random_state=SEED, verbose=-1)),
    }

scoring = {
    'roc_auc': 'roc_auc',
    'recall_pdac': 'recall',
    'specificity': make_scorer(recall_score, pos_label=0),
    'balanced_acc': 'balanced_accuracy',
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

rows = []
for fs_name, cols in [('Urine panel', URINE), ('Urine + CA19-9', WITH_CA199)]:
    for m_name, model in models().items():
        r = cross_validate(model, df[cols], y, cv=cv, scoring=scoring)
        row = {'features': fs_name, 'model': m_name}
        for k in scoring:
            v = r['test_' + k]
            row[k] = f'{v.mean():.3f} ± {v.std():.3f}'
        rows.append(row)
results = pd.DataFrame(rows)
results.to_csv('results.csv', index=False)
results

,features,model,roc_auc,recall_pdac,specificity,balanced_acc
0,Urine panel,Logistic regression,0.867 ± 0.033,0.762 ± 0.053,0.835 ± 0.045,0.798 ± 0.033
1,Urine panel,Random forest,0.884 ± 0.033,0.749 ± 0.038,0.838 ± 0.049,0.793 ± 0.029
2,Urine panel,LightGBM,0.870 ± 0.030,0.692 ± 0.069,0.842 ± 0.044,0.767 ± 0.046
3,Urine + CA19-9,Logistic regression,0.910 ± 0.030,0.806 ± 0.042,0.876 ± 0.036,0.841 ± 0.030
4,Urine + CA19-9,Random forest,0.945 ± 0.017,0.843 ± 0.059,0.906 ± 0.043,0.874 ± 0.029
5,Urine + CA19-9,LightGBM,0.939 ± 0.022,0.811 ± 0.078,0.904 ± 0.052,0.857 ± 0.043


## How large was the original leak?

The old protocol: impute on all data → duplicate every PDAC row → shuffle → 80/20 split. Same features as the original notebook (urine panel + CA19-9).

In [4]:
d = df[WITH_CA199 + ['y']].copy()
imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(d), columns=d.columns)
dup = pd.concat([imp, imp[imp['y'] == 1]], ignore_index=True).sample(frac=1, random_state=SEED)
Xtr, Xte, ytr, yte = train_test_split(dup[WITH_CA199], dup['y'], test_size=0.2, random_state=SEED)
seen = set(map(tuple, Xtr.values))
overlap = sum(tuple(r) in seen for r in Xte.values)
rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(Xtr, ytr)
p = rf.predict(Xte)
print(f'Test rows identical to a training row: {overlap} of {len(Xte)}')
print(f'Old-protocol random forest: recall {recall_score(yte, p):.3f}, accuracy {accuracy_score(yte, p):.3f}')

Test rows identical to a training row: 59 of 158
Old-protocol random forest: recall 0.972, accuracy 0.943


## Reading the results

- Compare the leak-free recall above with the old protocol's: the gap is the leak.
- Adding CA19-9 is a real experiment here: it changes performance even though it is missing for 40% of patients, so its contribution should be read alongside that missingness.
- Limitations: one dataset from one study, no external validation, default thresholds (no calibration or threshold tuning), small sample size. Debernardi et al. report their own validated model; this notebook does not compare against it.